In [ ]:
# Autograd is PyTorch’s automatic differentiation engine that tracks all tensor operations and
# computes gradients (derivatives) automatically using backpropagation.

# What is Autograd and Why It’s Needed

# What

# It records every operation you do on tensors that have requires_grad=True.

# Then it can automatically compute gradients using .backward().

# 🔹 Why

# In deep learning, we need gradients to update weights in training.

# Autograd saves you from manually calculating derivatives for complex models.





Autograd is PyTorch’s automatic differentiation engine that tracks all tensor operations and computes gradients (derivatives) automatically using backpropagation.

What is Autograd and Why It’s Needed

What

It records every operation you do on tensors that have requires_grad=True.

Then it can automatically compute gradients using .backward().

🔹 Why

In deep learning, we need gradients to update weights in training.

Autograd saves you from manually calculating derivatives for complex models.





In [ ]:
import torch
x=torch.tensor(2.0,requires_grad=True)
y = x**2 + 3*x + 2
y.backward()
print(x.grad)
# print(y.grad)7 → because dy/dx = 2x + 3 = 7

tensor(7.)


2️⃣ The Computation Graph


🔹 Definition

A graph is built dynamically by PyTorch when you perform tensor operations.

Each node is a tensor and each edge is an operation.

🔹 Dynamic Graph

Unlike TensorFlow (static graph), PyTorch builds the graph on the fly, making it easy to debug.


🔹 Example
x = torch.tensor(3.0, requires_grad=True)
y = x**3 + 2*x


Internal graph:

x → (power 3) → (+ 2x) → y


When you call .backward(), PyTorch traverses the graph backward using the chain rule to compute gradients.

requires_grad Flag
🔹 Purpose

It tells PyTorch to track all operations on the tensor for gradient computation.


🔹 Tip

If requires_grad=False, PyTorch ignores that tensor in the gradient graph.

In [ ]:

# 🔹 Example
a = torch.tensor(5.0, requires_grad=True)
b = a * 3
c = b + 2
c.backward()
print(a.grad)   # dc/da = 3

tensor(3.)


The .backward() Function

🔹 What it does

Triggers backpropagation.

Computes the gradient of the output (usually loss) w.r.t all tensors that have requires_grad=True.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = 3*x**2
y.backward()
print(x.grad)   # dy/dx = 6x = 12.    If the output is not scalar, you must pass a gradient argument to .backward().

tensor(12.)


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x * 2
y.backward(torch.tensor([1.0, 1.0, 1.0]))  # define gradient
print(x.grad)

tensor([2., 2., 2.])


In [ ]:
# Gradient Storage: .grad Attribute

# After calling .backward(), the gradient is stored in .grad for each leaf tensor.

# You can access it directly:

print(x.grad)


# Gradients accumulate, so clear them with:

x.grad.zero_()

tensor([2., 2., 2.])


tensor([0., 0., 0.])

Stop Tracking Gradients

 Sometimes you want to disable gradient tracking — for example, during model evaluation or inference.

 🔹 Methods:
(a) Using .detach()

Creates a new tensor detached from the graph.

In [ ]:
y = x.detach()
y

tensor([1., 2., 3.])

torch.no_grad()
🔹 Definition

torch.no_grad() is a context manager that disables gradient tracking temporarily.

🔹 Purpose

Used during:

Model evaluation / inference

Validation loops

Any operation where gradients are not required

Disabling autograd saves memory and computation.



In [ ]:
# 🔹 Syntax
# with torch.no_grad():
#     output = model(input)


x = torch.tensor(3.0, requires_grad=True)

with torch.no_grad():
    y = x * 2

print(y.requires_grad)   # False.  ✅ Inside torch.no_grad(), the computation graph is not created — faster and more efficient.

False


7️⃣ In-Place Operations and Gradients
⚠️ Warning:

In-place ops like x += 1 or x.mul_() can overwrite values needed for gradient computation.

PyTorch will raise:

RuntimeError: one of the variables needed for gradient computation has been modified by an inplace operation


Best Practice:
Avoid in-place ops when requires_grad=True.

In [ ]:
# Higher-Order Derivatives (Gradients of Gradients)

# PyTorch can compute second derivatives too:

x = torch.tensor(2.0, requires_grad=True)
y = x**3
dy_dx = torch.autograd.grad(y, x, create_graph=True)[0]
print(dy_dx)  # 6
dy2_dx2 = torch.autograd.grad(dy_dx, x)[0]
print(dy2_dx2)  # 6

tensor(12., grad_fn=<MulBackward0>)
tensor(12.)


Autograd in Neural Networks

During training:

Forward pass → compute loss

Backward pass → compute gradients

Optimizer step → update weights

26️⃣ Loss Functions
🔹 Definition

A loss function measures how far the model’s predictions are from the actual target.

Also called cost function / error function.

The goal: minimize loss using backpropagation.

🔹 Purpose / Logic

Output of loss → .backward() computes gradients w.r.t all model parameters.

Loss quantifies “how bad the model is”, guiding weight updates.

🔹 Examples

Mean Squared Error (MSE) for regression:

In [ ]:
import torch
import torch.nn as nn

y_pred = torch.tensor([2.5, 0.0], requires_grad=True)
y_true = torch.tensor([3.0, -0.5])

loss_fn = nn.MSELoss()
loss = loss_fn(y_pred, y_true)
loss.backward()

In [ ]:
# Cross-Entropy Loss for classification:

loss_fn = nn.CrossEntropyLoss()
output = torch.tensor([[2.0, 1.0], [0.5, 1.5]], requires_grad=True)
target = torch.tensor([0, 1])
loss = loss_fn(output, target)
loss.backward()

Optimizers
🔹 Definition

An optimizer updates model parameters (weights & biases) using computed gradients.

🔹 Purpose / Logic

Gradient tells direction to move, but we need learning rate & update rule.

Optimizer applies weight update rule:


🔹 Examples

SGD (Stochastic Gradient Descent)

In [ ]:
# Examples

# SGD (Stochastic Gradient Descent)

import torch.optim as optim

optimizer = optim.SGD(model.parameters(), lr=0.01)
optimizer.zero_grad()     # clear previous gradients
loss.backward()           # compute gradients
optimizer.step()          # update parameters

NameError: name 'model' is not defined

Adam (Adaptive Moment Estimation)
Better for deep networks:

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.001)


NameError: name 'model' is not defined

Training Loop
🔹 Definition

A loop that repeats forward + backward + optimizer step for multiple epochs to train the model.

🔹 Logic

Forward pass: compute predictions → y_pred = model(x)

Compute loss: loss = loss_fn(y_pred, y_true)

Backward pass: compute gradients → loss.backward()

Optimizer step: update weights → optimizer.step()

Reset gradients: optimizer.zero_grad() to avoid accumulation

In [ ]:
for epoch in range(100):
    optimizer.zero_grad()          # 1. reset gradients
    y_pred = model(x)              # 2. forward pass
    loss = loss_fn(y_pred, y_true) # 3. compute loss
    loss.backward()                # 4. backward pass
    optimizer.step()               # 5. update parameters
    print(f'Epoch {epoch}, Loss: {loss.item()}')


NameError: name 'optimizer' is not defined

Evaluation / Inference
🔹 Definition

Test model on new data to see how well it learned.

🔹 Logic

No gradient tracking → save memory

Use torch.no_grad() during evaluation